In [1]:
%load_ext autoreload
%autoreload 2
from SIDER_dataset.libraries.PCT_library import run_PCT
from pathlib import Path
import pandas as pd
from library import *
import os 

if os.name == "nt":
    projects_path = "C:\\Users\\KostasVoror\\Projects"
elif os.name == "posix":
    projects_path = "/Users/konkardatos/Projects"
clus_path = projects_path + "/clus/ClusProject/target/clus-2.12.8-deps.jar"
print(clus_path)
folder_path = r"/SIDER_dataset/datasets/binary_datasets"
print(folder_path)
measures = [
    "HammingLoss",
    "SubsetAccuracy",
    "RankingLoss",
    "MacroPrecision",
    "MacroRecall",
    "MacroFOne",
    "averageAUROC",
]

C:\Users\KostasVoror\Projects/clus/ClusProject/target/clus-2.12.8-deps.jar
C:\Users\KostasVoror\Projects\sep\SIDER_dataset\binary_datasets


In [2]:
results = {}

csv_files = [
    f for f in Path(folder_path).iterdir() if f.is_file() and f.suffix.lower() == ".csv"
]
for idx, file in enumerate(csv_files, start=1):
    print(f"{idx}/{len(csv_files)} - Processing {file.name}")
    dataset_name = file.name.replace(".csv", "")
    df = pd.read_csv(file)
    labels = get_ADRs(df)
    res = run_PCT(clus_path, str(file), labels, measures)
    results[dataset_name] = res

1/18 - Processing cpi_dataset_se_C0011603.csv
['java', '-jar', 'C:\\Users\\KostasVoror\\Projects\\clus\\ClusProject\\target\\clus-2.12.8-deps.jar', '-xval', 'C:\\Users\\KostasVoror\\Projects\\sep\\SIDER_dataset\\binary_datasets\\cpi_dataset_se_C0011603_settings.s']
STDOUT: Clus v2.12.8
Software for Predictive Clustering

Copyright (C) 2007 - 2025
   Katholieke Universiteit Leuven, Leuven, Belgium
   Jozef Stefan Institute, Ljubljana, Slovenia

This program is free software and comes with ABSOLUTELY NO
WARRANTY. You are welcome to redistribute it under certain
conditions. Type 'clus -copying' for distribution details.

[2025-04-27 10:42:34] [INFO] Loading 'cpi_dataset_se_C0011603_settings' 
[2025-04-27 10:42:34] [INFO] Reading ARFF Header 
[2025-04-27 10:42:34] [INFO] Reading CSV Data 
[2025-04-27 10:42:35] [INFO] Found 1377 rows 
[2025-04-27 10:42:35] [INFO] Space required by nominal attributes: 4 bytes/tuple regular, 0 bytes/tuple bitwise 
[2025-04-27 10:42:35] [INFO] Clustering attri

In [6]:
results

{'cpi_dataset_se_C0011603':            se_C0011603
 Precision     0.776536
 Recall        0.789773
 FOne          0.783099
 AUROC         0.559864,
 'cpi_dataset_se_C0012833':            se_C0012833
 Precision     0.759082
 Recall        0.790050
 FOne          0.774256
 AUROC         0.588610,
 'cpi_dataset_se_C0015230':            se_C0015230
 Precision     0.763254
 Recall        0.796756
 FOne          0.779645
 AUROC         0.539798,
 'cpi_dataset_se_C0018681':            se_C0018681
 Precision     0.795937
 Recall        0.796673
 FOne          0.796305
 AUROC         0.560101,
 'cpi_dataset_se_C0027497':            se_C0027497
 Precision     0.863519
 Recall        0.861301
 FOne          0.862409
 AUROC         0.594977,
 'cpi_dataset_se_C0042963':            se_C0042963
 Precision     0.800952
 Recall        0.801716
 FOne          0.801334
 AUROC         0.613526,
 'cpi_fingerprint_dataset_se_C0011603':            se_C0011603
 Precision     0.777365
 Recall        0.783178
 

In [22]:
results_df = pd.concat({k: v for k, v in results.items()}, axis=1)
results_df.columns = results_df.columns.get_level_values(0)

group_map = {
    "cpi_dataset": [],
    "fingerprint_dataset": [],
    "cpi_fingerprint_dataset": [],
}

for col in results_df.columns:
    if col.startswith("cpi_dataset"):
        group_map["cpi_dataset"].append(col)
    elif col.startswith("cpi_fingerprint_dataset"):
        group_map["cpi_fingerprint_dataset"].append(col)
    elif col.startswith("fingerprint_dataset"):
        group_map["fingerprint_dataset"].append(col)

results_df_macro = pd.DataFrame(
    {group: results_df[cols].mean(axis=1) for group, cols in group_map.items() if cols}
)

results_df = pd.concat([results_df_macro, results_df], axis=1)
path = "../datasets/binary_datasets/results/"
filename = "PCT_br_results"
save_df_to_csv(results_df, path, filename)
save_df_to_csv(results_df_macro, path, filename + "_Macro")
print(results_df.to_string())
results_df_macro

           cpi_dataset  fingerprint_dataset  cpi_fingerprint_dataset  cpi_dataset_se_C0011603  cpi_dataset_se_C0012833  cpi_dataset_se_C0015230  cpi_dataset_se_C0018681  cpi_dataset_se_C0027497  cpi_dataset_se_C0042963  cpi_fingerprint_dataset_se_C0011603  cpi_fingerprint_dataset_se_C0012833  cpi_fingerprint_dataset_se_C0015230  cpi_fingerprint_dataset_se_C0018681  cpi_fingerprint_dataset_se_C0027497  cpi_fingerprint_dataset_se_C0042963  fingerprint_dataset_se_C0011603  fingerprint_dataset_se_C0012833  fingerprint_dataset_se_C0015230  fingerprint_dataset_se_C0018681  fingerprint_dataset_se_C0027497  fingerprint_dataset_se_C0042963
Precision     0.793214             0.801534                 0.801884                 0.776536                 0.759082                 0.763254                 0.795937                 0.863519                 0.800952                             0.777365                             0.775369                             0.778182                             0.8

,cpi_dataset,fingerprint_dataset,cpi_fingerprint_dataset
Precision,0.793214,0.801534,0.801884
Recall,0.806045,0.830940,0.812480
FOne,0.799508,0.815945,0.807109
AUROC,0.576146,0.569319,0.578065
